# Civil War News — Notebook 3b: Regenerate Embeddings

Recomputes document embeddings and saves them to a **private Hugging Face dataset repo**.
Reads `news_proc` from Google Drive (same path as Notebook 3).

**Run once per model:**
1. Set `EMBEDDING_MODEL = 2` (MacBERTh) → Runtime > Run All.
2. Restart runtime → set `EMBEDDING_MODEL = 8` (bert_1850_1875) → Run All.

Encoding 8.4 M articles takes ~2 h on an A100 per model.

**First-time auth setup:** see Section 2.

In [ ]:
%pip install -q sentence-transformers datasets numpy huggingface_hub
print('Packages ready.')

In [ ]:
import os, zipfile, hashlib, gc, time
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer import modules as ST_models
from datasets import load_from_disk
from huggingface_hub import HfApi, login

# Keep-alive: reconnect every 60 s to prevent idle disconnect during long encoding.
try:
    from IPython.display import Javascript
    display(Javascript(
        'setInterval(() => { '
        'const btn = document.querySelector("colab-toolbar-button#connect"); '
        'if (btn) btn.click(); }, 60000)'
    ))
    print('Keep-alive active.')
except Exception:
    pass

if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {VRAM_GB:.1f} GB')
    DEVICE = 0
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')
    VRAM_GB = 0
    DEVICE  = -1

## Section 2: Configuration & Auth

**First-time setup** (once — saves tokens as Colab Secrets so future sessions are automatic):

**Hugging Face token:**
1. Go to huggingface.co → Settings → Access Tokens → New token → Role: **Write**.
2. Colab sidebar → **Secrets** → add secret named `HF_TOKEN` → paste the token.

**Google Drive:**
The Drive mount below will open a standard Google OAuth popup in your browser.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
EMBEDDING_MODEL = 2        # 2 = MacBERTh  |  8 = bert_1850_1875
USE_CHUNKING    = False    # True = chunk long articles, avg embeddings (~40% slower)
HF_REPO         = 'patrickjcrawford/civil-war-news'  # private dataset repo
DRIVE_BASE      = '/content/drive/MyDrive/CivilWarNews'  # path in your Google Drive
# ─────────────────────────────────────────────────────────────────────────────

LOCAL_BASE = '/content/data'

MODEL_CONFIGS = {
    2: {'model_id': 'emanjavacas/MacBERTh',
        'name': 'macberth',       'dim': 768,
        'doc_prefix': None,       'needs_manual_pooling': True,
        'encode_batch_size': 2048},
    8: {'model_id': 'Livingwithmachines/bert_1850_1875',
        'name': 'bert_1850_1875', 'dim': 768,
        'doc_prefix': None,       'needs_manual_pooling': True,
        'encode_batch_size': 2048},
}

CFG        = MODEL_CONFIGS[EMBEDDING_MODEL]
MODEL_NAME = CFG['name']

DRIVE_ZIP_PATH  = f'{DRIVE_BASE}/news_proc.zip'
NEWS_PROC_PATH  = f'{LOCAL_BASE}/news_proc'   # extracted to ephemeral storage
EMBEDDINGS_PATH = f'{LOCAL_BASE}/embeddings_{MODEL_NAME}.npy'
CACHE_KEY_PATH  = f'{LOCAL_BASE}/embeddings_{MODEL_NAME}.key'

_id  = CFG['model_id']
_dim = CFG['dim']
print(f'Model    : {_id} ({_dim}-dim)')
print(f'Name     : {MODEL_NAME}')
print(f'Chunking : {USE_CHUNKING}')
print(f'Input    : {NEWS_PROC_PATH}')
print(f'Output   : hf://datasets/{HF_REPO}/embeddings_{MODEL_NAME}.npy')

In [ ]:
# ── Hugging Face login ────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('[OK] Logged in to Hugging Face.')
except Exception as _e:
    print(f'HF login failed: {_e}')
    print('Add HF_TOKEN to Colab Secrets (see Section 2), or call login() interactively.')

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive mounted.')

## Section 3: Verify news_proc on Google Drive

In [ ]:
def _is_hf_dataset(path):
    return (os.path.exists(f'{path}/dataset_info.json') or
            os.path.exists(f'{path}/dataset_dict.json'))

os.makedirs(LOCAL_BASE, exist_ok=True)

if not _is_hf_dataset(NEWS_PROC_PATH):
    if os.path.exists(DRIVE_ZIP_PATH):
        print(f'Extracting {DRIVE_ZIP_PATH} → {LOCAL_BASE} ...')
        with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as _zf:
            _zf.extractall(LOCAL_BASE)
        print('Extraction complete.')
    else:
        raise FileNotFoundError(
            f'Dataset not found at {NEWS_PROC_PATH}\n'
            f'Also checked:  {DRIVE_ZIP_PATH}\n'
            f'Place news_proc.zip in your Drive under {DRIVE_BASE}/'
        )

if not _is_hf_dataset(NEWS_PROC_PATH):
    raise FileNotFoundError(f'Extraction finished but dataset still not found at {NEWS_PROC_PATH}')

print('[OK] news_proc ready')

In [ ]:
news_proc = load_from_disk(NEWS_PROC_PATH)
news_sub  = news_proc.shuffle(seed=42)
print(f'Loaded {len(news_sub):,} articles (Arrow-backed, not materialized)')

## Section 4: Compute Embeddings

~2 h on an A100 for 768-dim models at 8.4 M docs.  
The keep-alive cell above prevents idle disconnects.

In [ ]:
def load_embedding_model(cfg):
    model_id = cfg['model_id']
    if cfg['needs_manual_pooling']:
        word_model    = ST_models.Transformer(model_id, max_seq_length=512)
        pooling_model = ST_models.Pooling(
            word_model.get_word_embedding_dimension(),
            pooling_mode_mean_tokens=True,
        )
        model = SentenceTransformer(modules=[word_model, pooling_model], device='cuda')
    else:
        model = SentenceTransformer(model_id, device='cuda')
    if DEVICE != -1:
        model = model.half()
    print(f'Loaded: {model_id}  (dim={cfg["dim"]})')
    return model

def _get_chunks(text, tokenizer, max_len=510):
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_len:
        return [text]
    return [tokenizer.decode(ids[i:i + max_len], skip_special_tokens=True)
            for i in range(0, len(ids), max_len)]

def encode_docs(dataset, model, cfg, show_progress=True):
    base       = cfg['encode_batch_size']
    batch_size = min(base * 2, 8192) if VRAM_GB >= 30 else base
    chunk_size = batch_size * 16
    n_docs     = len(dataset)
    prefix     = cfg.get('doc_prefix')
    tokenizer  = model.tokenizer if USE_CHUNKING else None
    print(f'  docs={n_docs:,}  gpu_batch={batch_size}  chunking={USE_CHUNKING}')
    all_embs, t0, done = [], time.time(), 0
    with tqdm(total=n_docs, unit='doc', disable=not show_progress) as pbar:
        for batch in dataset.iter(batch_size=chunk_size):
            texts = batch['article']
            if prefix:
                texts = [prefix + t for t in texts]

            if not USE_CHUNKING:
                embs = model.encode(texts, batch_size=batch_size,
                                    show_progress_bar=False,
                                    convert_to_numpy=True,
                                    normalize_embeddings=True)
            else:
                # Split each article into 510-token chunks, encode flat, avg per article
                flat_chunks, article_ends = [], []
                for text in texts:
                    flat_chunks.extend(_get_chunks(text, tokenizer))
                    article_ends.append(len(flat_chunks))

                flat_embs = model.encode(flat_chunks, batch_size=batch_size,
                                         show_progress_bar=False,
                                         convert_to_numpy=True,
                                         normalize_embeddings=False)

                embs = np.zeros((len(texts), flat_embs.shape[1]), dtype=np.float32)
                prev = 0
                for i, end in enumerate(article_ends):
                    avg = flat_embs[prev:end].mean(axis=0)
                    norm = np.linalg.norm(avg)
                    embs[i] = avg / norm if norm > 0 else avg
                    prev = end

            all_embs.append(embs)
            torch.cuda.empty_cache()
            gc.collect()
            pbar.update(len(texts))
            done += len(texts)
            rate = done / (time.time() - t0)
            eta  = (n_docs - done) / rate if rate > 0 else 0
            pbar.set_postfix(docs_s=f'{rate:,.0f}', eta=f'{eta:.0f}s')
    return np.concatenate(all_embs)

embedding_model = load_embedding_model(CFG)

In [ ]:
# ── Encode (skips if local cache exists) ──────────────────────────────────────
if os.path.exists(EMBEDDINGS_PATH):
    print(f'Local cache found — loading {EMBEDDINGS_PATH}')
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f'Loaded shape: {embeddings.shape}')
else:
    print(f'Encoding {len(news_sub):,} articles with {CFG["model_id"]} ...')
    embeddings = encode_docs(news_sub, embedding_model, CFG)
    np.save(EMBEDDINGS_PATH, embeddings)
    _raw = f'{CFG["model_id"]}|512|{len(news_sub)}|42'
    with open(CACHE_KEY_PATH, 'w') as _f:
        _f.write(hashlib.md5(_raw.encode()).hexdigest())
    print(f'Saved locally: {EMBEDDINGS_PATH}  shape={embeddings.shape}')

# ── Upload to Hugging Face under MODEL_NAME subfolder ─────────────────────────
_api = HfApi()
_api.create_repo(HF_REPO, private=True, repo_type='dataset', exist_ok=True)
print(f'[OK] Repo ready: {HF_REPO}')

try:
    _existing = list(_api.list_repo_files(HF_REPO, repo_type='dataset'))
except Exception:
    _existing = []

_npy_repo_path = f'{MODEL_NAME}/embeddings_{MODEL_NAME}.npy'
if _npy_repo_path in _existing:
    print(f'[already on HF] {_npy_repo_path}')
else:
    _sz = os.path.getsize(EMBEDDINGS_PATH) / 1e9
    print(f'Uploading {_npy_repo_path} ({_sz:.1f} GB) ...')
    _api.upload_file(
        path_or_fileobj=EMBEDDINGS_PATH,
        path_in_repo=_npy_repo_path,
        repo_id=HF_REPO,
        repo_type='dataset',
    )
    print(f'Saved: hf://datasets/{HF_REPO}/{_npy_repo_path}')

_key_repo_path = f'{MODEL_NAME}/embeddings_{MODEL_NAME}.key'
if os.path.exists(CACHE_KEY_PATH) and _key_repo_path not in _existing:
    _api.upload_file(
        path_or_fileobj=CACHE_KEY_PATH,
        path_in_repo=_key_repo_path,
        repo_id=HF_REPO,
        repo_type='dataset',
    )
    print(f'Saved: hf://datasets/{HF_REPO}/{_key_repo_path}')

print(f'\nDone.  shape={embeddings.shape}  subfolder={MODEL_NAME}/')

## Done

Embeddings are saved to `hf://datasets/patrickjcrawford/civil-war-embeddings/`.

**To run the second model:**  
Runtime > Restart runtime → set `EMBEDDING_MODEL = 8` in the config cell → Run All.

**To download embeddings in Notebook 3:**
```python
from huggingface_hub import hf_hub_download
import numpy as np

path = hf_hub_download(
    repo_id='patrickjcrawford/civil-war-embeddings',
    filename='embeddings_macberth.npy',
    repo_type='dataset',
)
embeddings = np.load(path)
```